# Deep-Dive: High-Resolution Misprice Backtest

Two movies with clean minute/hour-level review timestamps in the critical window before bet close:
**forbidden_fruits_2026** and **they_will_kill_you**.

Unlike the main backtest (which aggregates reviews daily), this notebook builds the cumulative
score time series at **individual review resolution** -- every review arrival is a data point.
This lets us see exactly when edges appear, how long they persist, and whether
the market reacted.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from pathlib import Path
from glob import glob
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 110,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

ROOT = Path('.').resolve().parent  # notebooks/ -> project root
SLUGS = ['forbidden_fruits_2026', 'they_will_kill_you']
print(f"Project root: {ROOT}")
print(f"Target movies: {SLUGS}")

## 1. Load Data

In [ ]:
# ── Movies index (filtered to our two movies) ────────────────────────
mi = pd.read_csv(ROOT / 'movies_index.csv')

mi['volume'] = mi['Trading Volume ($)'].str.replace(r'[\$,]', '', regex=True).astype(float)
for col in ['Embargo Lift Date', 'Bet Open Date', 'Bet Close Date']:
    mi[col] = pd.to_datetime(mi[col], utc=True)

mi[['score_low', 'score_high']] = mi['Tomatometer Score Range Bet Close'].str.split('-', expand=True).astype(float)
mi['score_low_pct'] = mi['score_low'] * 100
mi['score_high_pct'] = mi['score_high'] * 100

mi[['reviews_low', 'reviews_high']] = mi['Total Reviews Bet Close'].str.split('-', expand=True).astype(float)
mi['reviews_T'] = mi['reviews_high']  # upper bound = conservative T

mi = mi[mi['Slug'].isin(SLUGS)].copy().reset_index(drop=True)
print(f"Movies loaded: {len(mi)}")
for _, r in mi.iterrows():
    print(f"  {r['Slug']}:")
    print(f"    Volume: ${r['volume']:,.0f}")
    print(f"    Bet close: {r['Bet Close Date'].strftime('%Y-%m-%d %H:%M UTC')}")
    print(f"    Score range: {r['score_low_pct']:.1f}% - {r['score_high_pct']:.1f}%")
    print(f"    Review count T (upper bound): {r['reviews_T']:.0f}")

In [ ]:
# ── Reviews (filtered) ────────────────────────────────────────────────
all_reviews = pd.read_csv(ROOT / 'reviews.csv')
all_reviews['estimated_timestamp'] = pd.to_datetime(all_reviews['estimated_timestamp'], utc=True, format='ISO8601')
all_reviews['is_fresh'] = (all_reviews['tomatometer_sentiment'] == 'positive').astype(int)

reviews = all_reviews[all_reviews['movie_slug'].isin(SLUGS)].copy()
reviews = reviews.sort_values(['movie_slug', 'estimated_timestamp']).reset_index(drop=True)

print(f"Reviews for target movies: {len(reviews)}")
for slug in SLUGS:
    m = reviews[reviews['movie_slug'] == slug]
    conf = m['timestamp_confidence'].value_counts().to_dict()
    print(f"  {slug}: {len(m)} reviews  (confidence: {conf})")
    print(f"    Date range: {m['estimated_timestamp'].min()} -- {m['estimated_timestamp'].max()}")
    print(f"    Fresh: {m['is_fresh'].sum()}, Rotten: {(1 - m['is_fresh']).sum()}")

In [ ]:
# ── Price histories (hour-level) ──────────────────────────────────────
def load_price_csv(movie_slug, freq='hour'):
    d = ROOT / 'rt-price-histories' / movie_slug
    if not d.exists():
        return None
    matches = list(d.glob(f'*-{freq}.csv'))
    if not matches:
        return None
    df = pd.read_csv(matches[0], parse_dates=['timestamp'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df = df.set_index('timestamp').sort_index()
    return df

price_data = {}
threshold_cols = {}
for slug in SLUGS:
    df = load_price_csv(slug, 'hour')
    if df is not None:
        price_data[slug] = df
        threshold_cols[slug] = sorted([c for c in df.columns if c.startswith('Above')],
                                       key=lambda c: int(c.split()[-1]))
        print(f"{slug}: {len(df)} hourly price rows, thresholds: {threshold_cols[slug]}")
        print(f"  Date range: {df.index.min()} -- {df.index.max()}")
    else:
        print(f"{slug}: NO price data found!")

## 2. Build Cumulative Score Time Series at Individual Review Resolution

Each review arrival is a data point. We sort reviews by `estimated_timestamp` and compute
running K (positive), N (total), then worst-case/best-case final score bounds using
T = upper bound of review count range.

In [ ]:
def build_review_level_series(slug, reviews_df, total_reviews_T):
    """Build cumulative score bounds at individual review resolution.
    
    Returns DataFrame indexed by estimated_timestamp with one row per review arrival,
    carrying running K, N, implied/best/worst case percentages.
    """
    movie_revs = reviews_df[reviews_df['movie_slug'] == slug].copy()
    if movie_revs.empty:
        return None
    
    movie_revs = movie_revs.sort_values('estimated_timestamp').reset_index(drop=True)
    
    # Cumulative counts
    movie_revs['cum_positive'] = movie_revs['is_fresh'].cumsum()       # K
    movie_revs['cum_total'] = np.arange(1, len(movie_revs) + 1)       # N
    
    T = total_reviews_T
    movie_revs['effective_T'] = np.maximum(movie_revs['cum_total'], T)
    movie_revs['reviews_remaining'] = np.maximum(movie_revs['effective_T'] - movie_revs['cum_total'], 0)
    
    movie_revs['implied_score_pct'] = (movie_revs['cum_positive'] / movie_revs['cum_total']) * 100
    movie_revs['best_case_pct'] = ((movie_revs['cum_positive'] + movie_revs['reviews_remaining']) / movie_revs['effective_T']) * 100
    movie_revs['worst_case_pct'] = (movie_revs['cum_positive'] / movie_revs['effective_T']) * 100
    
    return movie_revs

# Build for both movies
cum_series = {}
for _, row in mi.iterrows():
    slug = row['Slug']
    T = row['reviews_T']
    series = build_review_level_series(slug, reviews, T)
    if series is not None:
        cum_series[slug] = series
        final = series.iloc[-1]
        print(f"{slug}:")
        print(f"  Reviews: {int(final['cum_total'])} total, {int(final['cum_positive'])} positive")
        print(f"  T (upper bound): {T:.0f}")
        print(f"  Final implied score: {final['implied_score_pct']:.1f}%")
        print(f"  Final bounds: [{final['worst_case_pct']:.1f}%, {final['best_case_pct']:.1f}%]")
        print(f"  Reviews remaining (T - N): {int(final['reviews_remaining'])}")
        print()

## 3. Match Review Timestamps to Market Prices + Compute Edges

For each review arrival, look up the most recent hourly market price (forward-filled)
using `pd.merge_asof`. Then check lock status at each threshold and compute edges.

In [ ]:
def get_threshold_value(col_name):
    """Extract integer threshold from column name like 'Above 75'."""
    return int(col_name.split()[-1])

def resolve_threshold(threshold_val, score_low_pct, score_high_pct):
    """Actual resolution from movies_index score range.
    'Above X' resolves Yes if round(score_low_pct) >= X+1."""
    needed = threshold_val + 1
    if round(score_low_pct) >= needed:
        return 'Yes'
    elif round(score_high_pct) < needed:
        return 'No'
    else:
        return 'Ambiguous'

def check_lock(best_case_pct, worst_case_pct, threshold_val):
    """Check if outcome is locked from review bounds."""
    needed = threshold_val + 1
    if round(worst_case_pct) >= needed:
        return 'Yes'   # locked above threshold
    elif round(best_case_pct) < needed:
        return 'No'    # locked below threshold
    return None         # uncertain

# Build the edge time series at review-level resolution
edge_records = []

for _, mrow in mi.iterrows():
    slug = mrow['Slug']
    if slug not in cum_series or slug not in price_data:
        continue
    
    series = cum_series[slug]
    prices = price_data[slug]
    bet_close = mrow['Bet Close Date']
    score_low_pct = mrow['score_low_pct']
    score_high_pct = mrow['score_high_pct']
    
    # Forward-fill prices so every timestamp has a value
    prices_ffill = prices.ffill()
    
    for _, srow in series.iterrows():
        ts = srow['estimated_timestamp']
        hours_before_close = (bet_close - ts).total_seconds() / 3600
        
        if hours_before_close < 0:
            continue
        
        # Asof lookup: last price on or before this review timestamp
        mask = prices_ffill.index <= ts
        if not mask.any():
            price_row = {col: np.nan for col in threshold_cols[slug]}
        else:
            price_row = prices_ffill.loc[mask].iloc[-1]
        
        for col in threshold_cols[slug]:
            thresh_val = get_threshold_value(col)
            actual_resolution = resolve_threshold(thresh_val, score_low_pct, score_high_pct)
            if actual_resolution == 'Ambiguous':
                continue
            
            market_price = price_row[col] if col in price_row else np.nan
            if pd.isna(market_price):
                continue
            
            lock = check_lock(srow['best_case_pct'], srow['worst_case_pct'], thresh_val)
            
            if lock is not None and lock == actual_resolution:
                edge = (100 - market_price) if lock == 'Yes' else market_price
            else:
                edge = np.nan
            
            edge_records.append({
                'slug': slug,
                'threshold': thresh_val,
                'threshold_col': col,
                'actual_resolution': actual_resolution,
                'timestamp': ts,
                'hours_before_close': hours_before_close,
                'timestamp_confidence': srow['timestamp_confidence'],
                'cum_reviews': int(srow['cum_total']),
                'cum_positive': int(srow['cum_positive']),
                'reviews_remaining': int(srow['reviews_remaining']),
                'implied_score_pct': srow['implied_score_pct'],
                'best_case_pct': srow['best_case_pct'],
                'worst_case_pct': srow['worst_case_pct'],
                'lock': lock,
                'market_price': market_price,
                'edge_cents': edge,
            })

edges = pd.DataFrame(edge_records)
print(f"Edge observations: {len(edges):,}")
print(f"  Locked: {edges['lock'].notna().sum():,}")
print(f"  Locked with positive edge (>0c): {(edges['edge_cents'] > 0).sum():,}")

for slug in SLUGS:
    se = edges[edges['slug'] == slug]
    locked = se[se['lock'].notna()]
    mispriced = se[se['edge_cents'] > 0]
    print(f"\n  {slug}:")
    print(f"    Observations: {len(se)}, Locked: {len(locked)}, Mispriced (>0c): {len(mispriced)}")

## 4. Lock Event Table

For each threshold that locks, show when it first locked, how many reviews were in,
the market price at that moment, and the edge.

In [ ]:
locked_edges = edges[edges['lock'].notna()].copy()

for slug in SLUGS:
    mrow = mi[mi['Slug'] == slug].iloc[0]
    bet_close = mrow['Bet Close Date']
    slug_locked = locked_edges[locked_edges['slug'] == slug].copy()
    
    if slug_locked.empty:
        print(f"\n{'='*70}")
        print(f"{slug}: No thresholds locked")
        continue
    
    print(f"\n{'='*70}")
    print(f"{slug}")
    print(f"  Score range: {mrow['score_low_pct']:.1f}% - {mrow['score_high_pct']:.1f}%")
    print(f"  Bet close: {bet_close}")
    print(f"{'='*70}")
    
    # For each threshold, find the first lock event
    lock_events = []
    for thresh, grp in slug_locked.groupby('threshold'):
        grp = grp.sort_values('timestamp')
        first = grp.iloc[0]
        last = grp.iloc[-1]
        
        # Also get all positive-edge observations for this threshold
        pos_edge = grp[grp['edge_cents'] > 0]
        
        lock_events.append({
            'Threshold': f"Above {thresh}",
            'Resolution': first['actual_resolution'],
            'Lock Dir': first['lock'],
            'First Lock': first['timestamp'].strftime('%Y-%m-%d %H:%M'),
            'Hours Before Close': f"{first['hours_before_close']:.1f}",
            'Reviews In': int(first['cum_reviews']),
            'Reviews Left': int(first['reviews_remaining']),
            'Mkt Price (1st)': f"{first['market_price']:.0f}c",
            'Edge (1st)': f"{first['edge_cents']:.0f}c" if not pd.isna(first['edge_cents']) else '--',
            'Max Edge': f"{pos_edge['edge_cents'].max():.0f}c" if len(pos_edge) > 0 else '--',
            'Last Edge': f"{last['edge_cents']:.0f}c" if not pd.isna(last['edge_cents']) else '--',
        })
    
    lock_df = pd.DataFrame(lock_events)
    # Print as a formatted table
    print(lock_df.to_string(index=False))
    print()

## 5. Timeline Plots

For each movie, two subplots stacked vertically:

1. **Score bounds over time** -- best-case, worst-case, and implied score lines, with
   threshold levels marked as horizontal dashed lines. X-axis is hours before close.
2. **Edge in cents** for each locked threshold over time.

In [ ]:
THRESH_COLORS = plt.cm.tab10(np.linspace(0, 1, 11))  # up to 11 thresholds

for slug in SLUGS:
    mrow = mi[mi['Slug'] == slug].iloc[0]
    bet_close = mrow['Bet Close Date']
    series = cum_series[slug]
    slug_edges = edges[edges['slug'] == slug]
    
    # Compute hours before close for the review series
    series = series.copy()
    series['hours_before_close'] = (bet_close - series['estimated_timestamp']).dt.total_seconds() / 3600
    series = series[series['hours_before_close'] >= 0].copy()
    
    # Identify which thresholds have any data (locked or not)
    active_thresholds = sorted(slug_edges['threshold'].unique())
    # Color map for thresholds
    thresh_color_map = {t: THRESH_COLORS[i % len(THRESH_COLORS)] for i, t in enumerate(active_thresholds)}
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 11), sharex=True,
                                     gridspec_kw={'height_ratios': [2, 1]})
    
    # ── Top panel: Score bounds ──────────────────────────────────────
    ax1.fill_between(series['hours_before_close'], series['worst_case_pct'],
                     series['best_case_pct'], alpha=0.15, color='steelblue', label='Possible score range')
    ax1.plot(series['hours_before_close'], series['implied_score_pct'],
             color='steelblue', lw=1.5, label='Implied score (K/N)')
    ax1.plot(series['hours_before_close'], series['best_case_pct'],
             color='green', lw=1, ls='--', alpha=0.7, label='Best case')
    ax1.plot(series['hours_before_close'], series['worst_case_pct'],
             color='red', lw=1, ls='--', alpha=0.7, label='Worst case')
    
    # Threshold lines
    for t in active_thresholds:
        thresh_level = t + 0.5  # The "Above X" market: display the boundary
        ax1.axhline(thresh_level, color=thresh_color_map[t], ls=':', alpha=0.5, lw=0.8)
        ax1.text(series['hours_before_close'].max() * 0.99, thresh_level + 0.3,
                 f'Above {t}', fontsize=7, color=thresh_color_map[t], alpha=0.7, ha='right')
    
    # Final resolution range
    ax1.axhspan(mrow['score_low_pct'], mrow['score_high_pct'], alpha=0.08, color='gold',
                label=f"Final score range: {mrow['score_low_pct']:.1f}-{mrow['score_high_pct']:.1f}%")
    
    ax1.set_ylabel('Score (%)')
    ax1.set_title(f"{slug.replace('_', ' ').title()} -- Score Bounds at Review Resolution\n"
                  f"(T={mrow['reviews_T']:.0f}, Final: {mrow['score_low_pct']:.1f}-{mrow['score_high_pct']:.1f}%, "
                  f"Volume: ${mrow['volume']:,.0f})", fontsize=12)
    ax1.legend(loc='upper right', fontsize=8)
    ax1.grid(True, alpha=0.3)
    ax1.invert_xaxis()  # hours before close: rightmost = close
    
    # ── Bottom panel: Edge per threshold ─────────────────────────────
    slug_mispriced = slug_edges[slug_edges['edge_cents'] > 0]
    
    if slug_mispriced.empty:
        ax2.text(0.5, 0.5, 'No mispriced observations', transform=ax2.transAxes,
                 ha='center', va='center', fontsize=14, color='gray')
    else:
        for t in active_thresholds:
            t_data = slug_mispriced[slug_mispriced['threshold'] == t]
            if t_data.empty:
                continue
            ax2.scatter(t_data['hours_before_close'], t_data['edge_cents'],
                       color=thresh_color_map[t], s=30, alpha=0.7,
                       label=f"Above {t} ({t_data.iloc[0]['lock']})", edgecolor='k', linewidth=0.3)
            # Connect with line
            t_sorted = t_data.sort_values('hours_before_close', ascending=False)
            ax2.plot(t_sorted['hours_before_close'], t_sorted['edge_cents'],
                    color=thresh_color_map[t], lw=1, alpha=0.5)
    
    ax2.set_ylabel('Edge (cents)')
    ax2.set_xlabel('Hours Before Bet Close')
    ax2.set_title('Edge Over Time (locked thresholds only)')
    ax2.legend(loc='upper right', fontsize=8)
    ax2.grid(True, alpha=0.3)
    ax2.axhline(0, color='black', lw=0.5)
    
    plt.tight_layout()
    plt.savefig(ROOT / f'notebooks/{slug}_deep_dive.png', dpi=150, bbox_inches='tight')
    plt.show()
    print()

## 6. Market Price vs. Score Bounds — Zoomed to Last 72 Hours

Overlay the actual market prices for each threshold on top of the score bounds.
This shows whether the market tracked the converging bounds or lagged behind.

In [ ]:
for slug in SLUGS:
    mrow = mi[mi['Slug'] == slug].iloc[0]
    bet_close = mrow['Bet Close Date']
    prices = price_data[slug]
    series = cum_series[slug].copy()
    
    # Compute hours before close for prices
    prices_plot = prices.copy()
    prices_plot['hours_before_close'] = (bet_close - prices_plot.index).total_seconds() / 3600
    prices_plot = prices_plot[(prices_plot['hours_before_close'] >= 0) &
                              (prices_plot['hours_before_close'] <= 72)]
    
    # Review series in the same window
    series['hours_before_close'] = (bet_close - series['estimated_timestamp']).dt.total_seconds() / 3600
    series_window = series[(series['hours_before_close'] >= 0) & (series['hours_before_close'] <= 72)]
    
    # Identify thresholds near the final score
    active_thresholds = sorted([int(c.split()[-1]) for c in threshold_cols[slug]])
    # Focus on thresholds within 20pts of the final score
    final_mid = (mrow['score_low_pct'] + mrow['score_high_pct']) / 2
    focus_thresholds = [t for t in active_thresholds if abs(t - final_mid) <= 20]
    if not focus_thresholds:
        focus_thresholds = active_thresholds
    
    fig, ax = plt.subplots(figsize=(16, 8))
    
    # Score bounds
    if not series_window.empty:
        ax.fill_between(series_window['hours_before_close'], series_window['worst_case_pct'],
                         series_window['best_case_pct'], alpha=0.12, color='steelblue',
                         label='Score bounds (worst-best)')
        ax.plot(series_window['hours_before_close'], series_window['implied_score_pct'],
                 color='steelblue', lw=2, label='Implied score', zorder=5)
    
    # Market prices as thin lines (mapped to % scale)
    for i, t in enumerate(focus_thresholds):
        col = f'Above {t}'
        if col not in prices_plot.columns:
            continue
        color = THRESH_COLORS[i % len(THRESH_COLORS)]
        # Market price is probability in cents. "Above X" at price P means market thinks P% chance.
        # Plot the threshold level, and mark the market price along a secondary axis.
        series_data = prices_plot[['hours_before_close', col]].dropna()
        if series_data.empty:
            continue
        ax.plot(series_data['hours_before_close'], series_data[col],
                color=color, lw=1.2, alpha=0.8, ls='-',
                label=f"Mkt: {col} (cents)")
    
    # Threshold reference lines
    for t in focus_thresholds:
        ax.axhline(t + 0.5, color='gray', ls=':', alpha=0.3, lw=0.7)
    
    ax.axhspan(mrow['score_low_pct'], mrow['score_high_pct'], alpha=0.08, color='gold',
               label=f"Final: {mrow['score_low_pct']:.1f}-{mrow['score_high_pct']:.1f}%")
    
    ax.set_xlabel('Hours Before Bet Close')
    ax.set_ylabel('Score (%) / Market Price (cents)')
    ax.set_title(f"{slug.replace('_', ' ').title()} -- Last 72 Hours: Market Prices vs Score Bounds",
                 fontsize=12)
    ax.legend(loc='upper left', fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)
    ax.invert_xaxis()
    ax.set_ylim(0, 105)
    
    plt.tight_layout()
    plt.savefig(ROOT / f'notebooks/{slug}_last72h.png', dpi=150, bbox_inches='tight')
    plt.show()
    print()

## 7. Detailed Edge Evolution per Threshold

For each threshold that had a positive edge, show the full evolution:
review-by-review edge as reviews accumulate, with market price on a secondary axis.

In [ ]:
mispriced_edges = edges[edges['edge_cents'] > 0].copy()

for slug in SLUGS:
    slug_mp = mispriced_edges[mispriced_edges['slug'] == slug]
    thresholds_with_edge = sorted(slug_mp['threshold'].unique())
    
    if not thresholds_with_edge:
        print(f"\n{slug}: No thresholds with positive edge -- skipping detail plot.")
        continue
    
    n_thresh = len(thresholds_with_edge)
    fig, axes = plt.subplots(n_thresh, 1, figsize=(15, 4 * n_thresh), sharex=True)
    if n_thresh == 1:
        axes = [axes]
    
    mrow = mi[mi['Slug'] == slug].iloc[0]
    bet_close = mrow['Bet Close Date']
    
    for i, t in enumerate(thresholds_with_edge):
        ax = axes[i]
        t_edges = edges[(edges['slug'] == slug) & (edges['threshold'] == t)].sort_values('hours_before_close', ascending=False)
        t_locked = t_edges[t_edges['lock'].notna()]
        t_pos = t_edges[t_edges['edge_cents'] > 0]
        
        # Edge line (all locked observations)
        if not t_locked.empty:
            ax.fill_between(t_locked['hours_before_close'], 0, t_locked['edge_cents'].fillna(0),
                           alpha=0.2, color='green' if t_locked.iloc[0]['lock'] == 'Yes' else 'red')
            ax.plot(t_locked['hours_before_close'], t_locked['edge_cents'].fillna(0),
                   color='green' if t_locked.iloc[0]['lock'] == 'Yes' else 'red',
                   lw=1.5, marker='o', markersize=3, label=f"Edge (Lock={t_locked.iloc[0]['lock']})")
        
        # Market price on secondary y-axis
        ax2 = ax.twinx()
        ax2.plot(t_edges['hours_before_close'], t_edges['market_price'],
                color='purple', lw=1, alpha=0.6, ls='--', label='Market price')
        ax2.set_ylabel('Market Price (cents)', color='purple', fontsize=9)
        ax2.tick_params(axis='y', labelcolor='purple')
        
        # Annotations
        resolution = t_edges.iloc[0]['actual_resolution']
        lock_dir = t_locked.iloc[0]['lock'] if not t_locked.empty else '?'
        max_edge = t_pos['edge_cents'].max() if not t_pos.empty else 0
        first_lock_h = t_locked['hours_before_close'].max() if not t_locked.empty else None
        
        ax.set_ylabel('Edge (cents)')
        ax.set_title(f"Above {t}  |  Resolution: {resolution}  |  Lock: {lock_dir}  |  "
                     f"Max edge: {max_edge:.0f}c  |  First lock: {first_lock_h:.0f}h before close"
                     if first_lock_h else f"Above {t}", fontsize=10)
        ax.axhline(0, color='black', lw=0.5)
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper left', fontsize=8)
        ax2.legend(loc='upper right', fontsize=8)
        ax.invert_xaxis()
    
    axes[-1].set_xlabel('Hours Before Bet Close')
    fig.suptitle(f"{slug.replace('_', ' ').title()} -- Edge Evolution by Threshold", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(ROOT / f'notebooks/{slug}_edge_detail.png', dpi=150, bbox_inches='tight')
    plt.show()
    print()

## 8. Summary

Which thresholds locked, when, how big was the edge, and did the market correct?

In [ ]:
print("=" * 80)
print("DEEP-DIVE SUMMARY: HIGH-RESOLUTION MISPRICE BACKTEST")
print("=" * 80)

for slug in SLUGS:
    mrow = mi[mi['Slug'] == slug].iloc[0]
    bet_close = mrow['Bet Close Date']
    slug_edges = edges[edges['slug'] == slug]
    slug_locked = slug_edges[slug_edges['lock'].notna()]
    slug_mispriced = slug_edges[slug_edges['edge_cents'] > 0]
    
    print(f"\n{'─'*80}")
    print(f"  {slug.replace('_', ' ').upper()}")
    print(f"  Volume: ${mrow['volume']:,.0f}  |  Final score: {mrow['score_low_pct']:.1f}-{mrow['score_high_pct']:.1f}%  |  T={mrow['reviews_T']:.0f}")
    print(f"  Bet close: {bet_close}")
    print(f"{'─'*80}")
    
    series = cum_series[slug]
    n_reviews = len(series)
    n_mh = len(series[series['timestamp_confidence'].isin(['m', 'h'])])
    print(f"  Total reviews: {n_reviews}  (minute/hour precision: {n_mh})")
    
    if slug_mispriced.empty:
        print(f"  ** No mispriced thresholds found **")
        # Still show what locked
        if not slug_locked.empty:
            print(f"  Locked thresholds (no edge, market was correct):")
            for t in sorted(slug_locked['threshold'].unique()):
                t_data = slug_locked[slug_locked['threshold'] == t]
                first = t_data.sort_values('timestamp').iloc[0]
                print(f"    Above {t}: locked {first['lock']} at {first['hours_before_close']:.0f}h before close, "
                      f"market was {first['market_price']:.0f}c (edge: {first['edge_cents']:.0f}c)")
        continue
    
    print(f"  Mispriced thresholds: {sorted(slug_mispriced['threshold'].unique())}")
    print()
    
    for t in sorted(slug_mispriced['threshold'].unique()):
        t_all = slug_edges[slug_edges['threshold'] == t].sort_values('hours_before_close', ascending=False)
        t_locked = t_all[t_all['lock'].notna()]
        t_pos = t_all[t_all['edge_cents'] > 0]
        
        first_lock = t_locked.iloc[0] if not t_locked.empty else None
        last_lock = t_locked.iloc[-1] if not t_locked.empty else None
        max_edge_row = t_pos.loc[t_pos['edge_cents'].idxmax()] if not t_pos.empty else None
        
        # Did market correct? Compare first and last edge
        first_edge = first_lock['edge_cents'] if first_lock is not None and not pd.isna(first_lock['edge_cents']) else None
        last_edge = last_lock['edge_cents'] if last_lock is not None and not pd.isna(last_lock['edge_cents']) else None
        
        if first_edge is not None and last_edge is not None and first_edge > 0:
            correction = first_edge - last_edge
            pct_corrected = (correction / first_edge) * 100 if first_edge > 0 else 0
            correction_str = f"  Market corrected {pct_corrected:.0f}% ({first_edge:.0f}c -> {last_edge:.0f}c)"
        else:
            correction_str = "  No correction data"
        
        print(f"  Above {t}:")
        print(f"    Resolution: {first_lock['actual_resolution'] if first_lock is not None else '?'}")
        print(f"    First locked: {first_lock['hours_before_close']:.1f}h before close "
              f"({first_lock['cum_reviews']} reviews in, {first_lock['reviews_remaining']} remaining)"
              if first_lock is not None else "    No lock")
        print(f"    Max edge: {max_edge_row['edge_cents']:.0f}c at {max_edge_row['hours_before_close']:.1f}h "
              f"before close (market: {max_edge_row['market_price']:.0f}c)"
              if max_edge_row is not None else "    No positive edge")
        print(f"    Last observation: {last_lock['hours_before_close']:.1f}h before close, "
              f"edge={last_lock['edge_cents']:.0f}c, market={last_lock['market_price']:.0f}c"
              if last_lock is not None else "")
        print(f"   {correction_str}")
        print()

print(f"\n{'='*80}")
print("Key question: For these two movies with minute/hour review timestamps,")
print("do the locked edges appear with enough lead time to trade profitably?")
print("=" * 80)